# Module 4 — Exercise: Beaconing Analysis & Autoencoder Detection

This notebook uses your Zeek exports (`http.log` and `weird.log`) to:

1. Demonstrate periodic beaconing via time-series, FFT, and autocorrelation.
2. Train a small sequence autoencoder (LSTM) to detect beacon-like sequences.


## 1) Load logs and show columns

We expect `http.log` and `weird.log` as JSON-lines (each line a JSON object).

In [1]:
import json, os, pandas as pd, numpy as np
HTTP_PATH = "./http.log"
WEIRD_PATH = "./weird.log"

def load_jsonlines(path, max_lines=None):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line=line.strip()
            if not line: 
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                # skip bad lines but print a warning
                print(f"Warning: failed to parse line {i} in {path}: {e}")
            if max_lines and len(rows) >= max_lines:
                break
    return pd.DataFrame(rows)

print('Loading http.log ...')
df_http = load_jsonlines(HTTP_PATH)
print('http rows,cols:', df_http.shape)
print('\nColumns (http.log):')
print(df_http.columns.tolist())

print('\nLoading weird.log ...')
df_weird = load_jsonlines(WEIRD_PATH)
print('weird rows,cols:', df_weird.shape)
print('\nColumns (weird.log):')
print(df_weird.columns.tolist())

# quick preview
df_http_head = df_http.head(5)
df_weird_head = df_weird.head(5)
df_http_head, df_weird_head

Loading http.log ...
http rows,cols: (36, 20)

Columns (http.log):
['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'trans_depth', 'method', 'host', 'uri', 'version', 'user_agent', 'request_body_len', 'response_body_len', 'status_code', 'status_msg', 'tags', 'referrer', 'resp_fuids', 'resp_mime_types']

Loading weird.log ...
weird rows,cols: (7, 10)

Columns (weird.log):
['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'name', 'notice', 'peer', 'source']


(             ts                 uid     id.orig_h  id.orig_p     id.resp_h  \
 0  1.761358e+09  CmWiZT3rlLNJCTcGog  192.168.56.2      33368  192.168.56.3   
 1  1.761358e+09  C0hUzR32Ruw6g4dYX2  192.168.56.2      33384  192.168.56.3   
 2  1.761360e+09   CBB3VNmYQUE1JfNw3  192.168.56.2      51984  192.168.56.3   
 3  1.761360e+09   CMIMd3M4r6G73cTq7  192.168.56.2      36958  192.168.56.3   
 4  1.761360e+09  CNU53Z1ddcBrtiNXjg  192.168.56.2      53380  192.168.56.3   
 
    id.resp_p  trans_depth method               host           uri version  \
 0       8000            1    GET  192.168.56.3:8000             /     1.0   
 1       8000            1    GET  192.168.56.3:8000  /favicon.ico     1.0   
 2       8000            1    GET  192.168.56.3:8000             /     1.0   
 3       8000            1    GET  192.168.56.3:8000             /     1.0   
 4       8000            1    GET  192.168.56.3:8000             /     1.0   
 
                                           user_agent 

## 2) Descriptive statistics

Numeric summaries and top categorical values for important fields.

In [2]:
import numpy as np
# Timestamp parsing
for df, name in [(df_http, 'http'), (df_weird, 'weird')]:
    if 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], unit='s', origin='unix', errors='coerce')
        print(f"{name}: time range {df['ts'].min()} -> {df['ts'].max()}")
    else:
        print(f"{name}: no 'ts' column found")

# Numeric describe for http
num_http = df_http.select_dtypes(include=[np.number]).columns.tolist()
print('\nNumeric columns in http.log:', num_http)
if num_http:
    display(df_http[num_http].describe().T)

# Top values for key categorical columns
for col in ['host','uri','method','user_agent','status_code']:
    if col in df_http.columns:
        print(f"\nTop values for http.{col}:")
        display(df_http[col].astype(str).value_counts().head(10))

http: time range 2025-10-25 02:12:49.686234951 -> 2025-10-25 02:55:54.010204077
weird: time range 2025-10-24 09:25:46.443435907 -> 2025-10-25 02:42:35.275976896

Numeric columns in http.log: ['id.orig_p', 'id.resp_p', 'trans_depth', 'request_body_len', 'response_body_len', 'status_code']


,count,mean,std,min,25%,50%,75%,max
id.orig_p,36.0,44976.055556,7327.274582,32930.0,38564.5,45171.0,50430.0,58860.0
id.resp_p,36.0,8000.000000,0.000000,8000.0,8000.0,8000.0,8000.0,8000.0
trans_depth,36.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
request_body_len,36.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
response_body_len,36.0,12.777778,76.666667,0.0,0.0,0.0,0.0,460.0
status_code,36.0,205.666667,34.000000,200.0,200.0,200.0,200.0,404.0



Top values for http.host:


host
192.168.56.3:8000    36
Name: count, dtype: int64


Top values for http.uri:


...

method
GET    36
Name: count, dtype: int64


Top values for http.user_agent:


user_agent
python-requests/2.32.4                                                    34
Mozilla/5.0 (X11; Linux x86_64; rv:128.0) Gecko/20100101 Firefox/128.0     2
Name: count, dtype: int64


Top values for http.status_code:


status_code
200    35
404     1
Name: count, dtype: int64

## 3) Build time series and inter-arrival times (host: 192.168.56.2)

We will focus on id.orig_h == '192.168.56.2' (attacker) making HTTP GETs to 192.168.56.3:8000.
Compute counts per second and inter-arrival (delta) times.

In [3]:
TARGET_HOST = "192.168.56.2"
# filter HTTP rows from target host
http_src = df_http[df_http.get('id.orig_h') == TARGET_HOST].copy()
print('Filtered http rows from target:', http_src.shape[0])
http_src = http_src.sort_values('ts')

# counts per second / per minute
http_src['ts_sec'] = http_src['ts'].dt.floor('s')
counts_sec = http_src.groupby('ts_sec').size().rename('count').reset_index()

http_src['iat'] = http_src['ts'].diff().dt.total_seconds()
counts_sec.head(10), http_src[['ts','id.orig_p','uri','iat']].head(10)

Filtered http rows from target: 36


(               ts_sec  count
 0 2025-10-25 02:12:49      2
 1 2025-10-25 02:37:48      1
 2 2025-10-25 02:38:44      1
 3 2025-10-25 02:39:40      1
 4 2025-10-25 02:40:35      1
 5 2025-10-25 02:41:33      1
 6 2025-10-25 02:41:37      1
 7 2025-10-25 02:42:35      1
 8 2025-10-25 02:42:40      1
 9 2025-10-25 02:43:36      1,
                              ts  id.orig_p           uri          iat
 0 2025-10-25 02:12:49.686234951      33368             /          NaN
 1 2025-10-25 02:12:49.918461084      33384  /favicon.ico     0.232226
 2 2025-10-25 02:37:48.476722956      51984             /  1498.558262
 3 2025-10-25 02:38:44.878037930      36958             /    56.401315
 4 2025-10-25 02:39:40.286370993      53380             /    55.408333
 5 2025-10-25 02:40:35.548990011      32930             /    55.262619
 6 2025-10-25 02:41:33.249766111      43902             /    57.700776
 7 2025-10-25 02:41:37.122688055      43910             /     3.872922
 8 2025-10-25 02:42:35.2807939

## 4) Fourier Transform (FFT) and Auto-correlation

FFT on counts-per-second to find dominant periodicities, and autocorrelation on the inter-arrival series.

In [4]:
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from statsmodels.tsa.stattools import acf

# prepare counts vector (fill missing seconds with zeros)
counts_ts = counts_sec.set_index('ts_sec').resample('1s').sum().fillna(0)['count']
signal = counts_ts.values
N = len(signal)
results = {}

if N < 8:
    results['fft'] = 'Not enough points for FFT -- need more captured time.'
else:
    yf = fft(signal - signal.mean())
    xf = fftfreq(N, d=1.0)  # seconds resolution
    power = np.abs(yf)
    pos = xf > 0
    # prepare plot data as lists (can't display interactive plot here)
    results['freqs'] = xf[pos].tolist()
    results['power'] = power[pos].tolist()

# Autocorrelation of iat (drop NaNs)
iat = http_src['iat'].dropna().values
if len(iat) < 5:
    results['acf'] = 'Not enough iat samples for ACF'
else:
    corr = acf(iat, nlags=min(200, len(iat)-1))
    results['acf'] = corr.tolist()

results['N'] = N
results

{'freqs': [0.0003866976024748647,
  0.0007733952049497294,
  0.001160092807424594,
  0.0015467904098994587,
  0.0019334880123743235,
  0.002320185614849188,
  0.0027068832173240526,
  0.0030935808197989174,
  0.003480278422273782,
  0.003866976024748647,
  0.004253673627223511,
  0.004640371229698376,
  0.005027068832173241,
  0.005413766434648105,
  0.0058004640371229705,
  0.006187161639597835,
  0.006573859242072699,
  0.006960556844547564,
  0.007347254447022429,
  0.007733952049497294,
  0.008120649651972157,
  0.008507347254447023,
  0.008894044856921888,
  0.009280742459396751,
  0.009667440061871617,
  0.010054137664346482,
  0.010440835266821347,
  0.01082753286929621,
  0.011214230471771076,
  0.011600928074245941,
  0.011987625676720804,
  0.01237432327919567,
  0.012761020881670535,
  0.013147718484145398,
  0.013534416086620264,
  0.013921113689095129,
  0.014307811291569992,
  0.014694508894044857,
  0.015081206496519723,
  0.015467904098994588,
  0.01585460170146945,
  0

## 5) Heatmap: periodic vs noisy signals across hosts

Create a heatmap (hosts x time) of counts per minute to visually inspect which hosts show periodic activity.

In [5]:
# Build multi-host per-minute pivot table for visualization (use http.log)
df_http['ts_min'] = df_http['ts'].dt.floor('min')
pivot = df_http.groupby(['id.orig_h','ts_min']).size().rename('count').reset_index()
heat = pivot.pivot(index='id.orig_h', columns='ts_min', values='count').fillna(0)
# reduce to top hosts by total volume
host_totals = heat.sum(axis=1).sort_values(ascending=False)
top_hosts = host_totals.head(25).index.tolist()
heat_sub = heat.loc[top_hosts]

# provide summary stats
summary = {
    'num_hosts': len(host_totals),
    'top_hosts': top_hosts,
    'heat_shape': heat_sub.shape
}
summary

{'num_hosts': 1, 'top_hosts': ['192.168.56.2'], 'heat_shape': (1, 20)}

## 6) Sequence Autoencoder (LSTM) — prepare windows

Prepare sliding windows of inter-arrival times for the target host. We will train a small LSTM autoencoder in the next cell.

In [6]:
# Prepare sliding windows of iat for target host
WINDOW = 16  # small window for classroom demo
STEP = 1

series = http_src['iat'].fillna(0).values  # fill NaN with 0 for start
# build windows
X = []
for start in range(0, max(1, len(series)-WINDOW+1), STEP):
    win = series[start:start+WINDOW]
    if len(win) == WINDOW:
        X.append(win)
X = np.array(X)
X_shape = X.shape
X_shape

(21, 16)

In [7]:
# Normalize and split (if windows exist)
from sklearn.preprocessing import StandardScaler
if X.shape[0] > 0:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.reshape(-1, WINDOW)).reshape(-1, WINDOW, 1)
    n = len(X_scaled)
    split = max(1, int(n * 0.7))
    X_train = X_scaled[:split]
    X_test = X_scaled[split:]
    shapes = {'X_train': X_train.shape, 'X_test': X_test.shape}
else:
    X_scaled = None
    X_train = None
    X_test = None
    shapes = 'no windows'
shapes

{'X_train': (14, 16, 1), 'X_test': (7, 16, 1)}

## 7) Sequence Autoencoder (LSTM) — train & evaluate (if windows exist)

A tiny LSTM autoencoder will be trained — this is for demonstration and may require TensorFlow installed.

In [9]:
# Check kernel Python path
import sys
print(sys.executable)
# Install TensorFlow into that exact interpreter (runs in notebook cell)
!"{sys.executable}" -m pip install --upgrade pip
!"{sys.executable}" -m pip install tensorflow

f:\anaconda\python.exe
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 4.9 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.11.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/331.9 MB 4.2 MB/s eta 0:01:20
   -----------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tensorflow]
   ------------------------------------- -- 15/16 [tens

In [11]:
# ...existing code...
trained = False
if X_train is not None and X_train.shape[0] > 3:
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Model
        from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
        tf.random.set_seed(42)
        timesteps = WINDOW
        features = 1
        latent_dim = 8
        inputs = Input(shape=(timesteps, features))
        encoded = LSTM(latent_dim, activation='tanh')(inputs)
        decoded = RepeatVector(timesteps)(encoded)
        decoded = LSTM(latent_dim, activation='tanh', return_sequences=True)(decoded)
        outputs = TimeDistributed(Dense(features))(decoded)
        autoencoder = Model(inputs, outputs)
        autoencoder.compile(optimizer='adam', loss='mse')
        history = autoencoder.fit(X_train, X_train, epochs=12, batch_size=16, validation_split=0.1, verbose=0)
        X_test_pred = autoencoder.predict(X_test)
        import numpy as np
        mse = np.mean((X_test_pred - X_test)**2, axis=(1,2))
        import statistics
        threshold = statistics.mean(mse) + 3*statistics.pstdev(mse)
        trained = True
        eval_res = {'mse_mean': float(mse.mean()), 'mse_std': float(mse.std()), 'threshold_suggested': float(threshold), 'mse_count': int(len(mse))}
    except ModuleNotFoundError as e:
        # actionable message pointing to the exact interpreter to install into
        import sys
        install_cmd = f'"{sys.executable}" -m pip install tensorflow'
        eval_res = {
            'error': str(e),
            'note': 'TensorFlow not found in the current kernel environment.',
            'install_command_for_kernel_python': install_cmd,
            'next_steps': 'Run the install command in the notebook or terminal, then restart the kernel.'
        }
    except Exception as e:
        eval_res = {'error': str(e)}
else:
    eval_res = 'Not enough windows to train (need >3 train windows)'
trained, eval_res
# ...existing code...

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 505ms/step


(True,
 {'mse_mean': 0.42973379808243173,
  'mse_std': 0.09636700454734617,
  'threshold_suggested': 0.7188348117244703,
  'mse_count': 7})

## 8) Narrative: tying this exercise back to the presentation

- **FFT & Auto-correlation:** used to reveal periodicities in HTTP counts and repeated inter-arrival peaks. These correspond to beacon intervals discussed in the slides.

- **Sequence Autoencoder:** LSTM learns normal timing patterns and flags high reconstruction error for unusual periodic sequences. Adjust threshold (mean + k*std) as an exercise to trade off false positives vs detection.

- **Exercise prompts:**
  1. Change `WINDOW` and observe impact on detection sensitivity.
  2. Use `beacon_lab.py` with different jitter settings and re-run capture → see how FFT/ACF and autoencoder respond.
  3. Export flagged windows and craft a Sigma rule template that uses periodicity (e.g., repeated GETs at ~T seconds) + high AE reconstruction error.
